In [ ]:
%matplotlib widget
import read_h5
import magic_graphs


In [3]:
file_path = r'/Users/iuriikibalin/Downloads/sim_c60/mccode.h5'

dg_magic = read_h5.read_magic_from_nexus(file_path)
da_detector = dg_magic['detector_a']
da_detector = da_detector.transform_coords(('event_gamma', 'event_position_global'), graph=magic_graphs.graph_detector, rename_dims=False)

In [ ]:
points_toa = 200
data_hist_3d = da_detector.hist(toa=points_toa)

In [ ]:
data_hist_3d = data_hist_3d.assign_coords(**{
    'event_gamma':data_hist_3d.coords['event_gamma'].to(unit="deg"),
    'event_nu':data_hist_3d.coords['event_nu'].to(unit="deg"),
    })

In [ ]:
n_shift = 4

for i in range(points_toa):
    toa_min = data_hist_3d.coords['toa'][i]
    toa_max = data_hist_3d.coords['toa'][i+1]
    delta_time = toa_max - toa_min
    for j in range(n_shift):
        ii = j + n_shift*i
        print(f'Progress {100*(ii+1)/(points_toa*n_shift):1.1f}%', end="\r")
        da_time = data_hist_3d['toa', toa_min+delta_time*(j/n_shift): toa_min + delta_time*(1+j/n_shift)]
        bin_nu = scipp.linspace('event_nu', -35, 15, num=300, unit='deg', endpoint=True)
        bin_gamma = scipp.linspace('event_gamma', 10, 80, num=1000, unit='deg', endpoint=True)
        fig = da_time.hist(event_nu=bin_nu, event_gamma=bin_gamma).sum('toa').plot(vmax=0.05)


        # Render current frame to PNG bytes
        fig.save(f'frame_{ii:04d}.png')  # Plopp figure → PNG bytes

# ffmpeg -r 30 -i frame_%04d.png -vcodec libx264 animation.mp4    